# Solar System Object Fink Table

This service lets you query the table containing aggregated parameters for known Solar System objects in Fink. This table is updated once a month, with all data in Fink.

```{note} Starting date: 2026/04/04
The SSOFT uses LSST alert data only from 2026/04/04. Prior to that date, the project was not providing correct units for the topocentric and heliocentric distance fields that are used to fit parameters for some models.
````

_Created on 2026/09/28. See https://doc.lsst.fink-broker.org/services/api/solar_system/ for up-to-date documentation._

In [1]:
import io

import pandas as pd
import requests

# Phase curve model, among: HG, HG1G2, SHG1G2
flavor = "HG"

r = requests.post(
    "https://api.lsst.fink-portal.org/api/v1/ssoft",
    json={"flavor": flavor, "output-format": "parquet"},
)

# Format output in a DataFrame
ssoft_lsst = pd.read_parquet(io.BytesIO(r.content))

This table contains basic statistics (e.g. coverage in time for each object, name, number, ...), fitted parameters (absolute magnitude, phase parameters, spin parameters, ...), quality statuses, and version numbers. 

In [2]:
ssoft_lsst.head(2)

,designation,chi2red,status,fit,rms,rms_i,rms_r,rms_z,median_error_phot,median_error_phot_i,...,max_phase_y,H_y,err_H_y,G_y,err_G_y,n_days_y,sso_name,sso_number,version,flag
0,2013 CL29,52.143506,1,0,0.062664,0.085624,0.0,0.026386,0.026739,0.024591,...,NaN,NaN,NaN,NaN,NaN,NaN,2013 CL29,451695,202609,0
1,2015 PY171,10.304822,2,0,0.111956,0.066456,0.0,0.173901,0.056465,0.049532,...,NaN,NaN,NaN,NaN,NaN,NaN,2015 PY171,822188,202609,0


In [3]:
# Number of objects passing the criteria to enter phase curve measurement
len(ssoft_lsst)

58163

````{warning} Table size
The table is several megabytes by default (and as the survey will progress, it will become several hundreds of megabytes), so you can also decide to transfer only a subset of fields:
```python
import requests

r = requests.post(
    "https://api.lsst.fink-portal.org/api/v1/ssoft",
    json={"columns": "sso_name,H_g,chi2red", "output-format": "parquet"},
)
```
````

To know the parameters of interest, you can retrieve the schema of the table using using the schema argument:

In [4]:
import requests

r = requests.post(
    "https://api.lsst.fink-portal.org/api/v1/ssoft",
    json={"flavor": flavor, "schema": True},
)

schema = r.json()

````{warning} Data quality
As of 2026/09, the survey is beginning. The lightcurve sampling for each object is not yet sufficient, and phase coverage is small. Do not expect ground breaking results at this stage!
````

Let's apply some quality cuts:

In [5]:
c1 = ssoft_lsst["n_obs"] >= 20  # at least 20 observations
c2 = ssoft_lsst["n_days"] >= 14  # observed for at least 2 weeks
c3 = ssoft_lsst["chi2red"] < 10  # avoid crazy reduced chi2

ssoft_lsst_clean = ssoft_lsst[c1 & c2 & c3]

In [6]:
ssoft_lsst_clean.sort_values("n_obs", ascending=False)

,designation,chi2red,status,fit,rms,rms_i,rms_r,rms_z,median_error_phot,median_error_phot_i,...,max_phase_y,H_y,err_H_y,G_y,err_G_y,n_days_y,sso_name,sso_number,version,flag
16512,2003 BU55,4.997225,2,0,0.049782,0.015696,0.020872,0.024139,0.009121,0.006784,...,NaN,NaN,NaN,NaN,NaN,NaN,2003 BU55,174505,202609,0
34717,2002 VF21,9.532202,2,0,0.066620,0.040291,0.026886,0.041671,0.014443,0.011262,...,NaN,NaN,NaN,NaN,NaN,NaN,2002 VF21,170032,202609,0
37770,2017 SZ102,4.599290,2,0,0.086021,0.093004,0.061906,0.087236,0.042538,0.036399,...,NaN,NaN,NaN,NaN,NaN,NaN,2017 SZ102,540525,202609,0
24711,2006 OA12,1.852437,4,0,0.062368,0.050422,0.056540,0.092051,0.043363,0.040301,...,NaN,NaN,NaN,NaN,NaN,NaN,2006 OA12,319466,202609,0
8044,1998 TT26,3.561930,3,0,0.027610,0.009423,0.009891,0.012465,0.006253,0.004921,...,NaN,NaN,NaN,NaN,NaN,NaN,1998 TT26,25223,202609,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52799,2021 ES60,1.470383,1,0,0.078809,0.075519,0.083210,0.134614,0.085044,0.085228,...,NaN,NaN,NaN,NaN,NaN,NaN,2021 ES60,nan,202609,0
14472,2019 OB24,2.862863,4,0,0.114405,0.089921,0.131394,0.000000,0.100149,0.111911,...,NaN,NaN,NaN,NaN,NaN,NaN,2019 OB24,nan,202609,0
32929,2018 PX75,3.378440,1,0,0.135538,0.080307,0.192152,NaN,0.100563,0.100862,...,NaN,NaN,NaN,NaN,NaN,NaN,2018 PX75,nan,202609,0
8777,2010 JU2,2.086813,2,0,0.075390,0.064764,0.094691,0.054870,0.090526,0.061328,...,NaN,NaN,NaN,NaN,NaN,NaN,2010 JU2,552655,202609,0


## Adding another survey: ZTF

In [7]:
import io

import pandas as pd
import requests

# Phase curve model, among: HG, HG1G2, SHG1G2
flavor = "HG"

r = requests.post(
    "https://api.ztf.fink-portal.org/api/v1/ssoft",
    json={"flavor": flavor, "output-format": "parquet"},
)

# Format output in a DataFrame
ssoft_ztf = pd.read_parquet(io.BytesIO(r.content))

In [8]:
cols = ["sso_name", "n_obs", "n_days"]
group = (
    ssoft_lsst_clean[cols]
    .set_index("sso_name")
    .join(
        ssoft_ztf[cols].set_index("sso_name"),
        on="sso_name",
        how="inner",
        rsuffix="_ztf",
        lsuffix="_lsst",
    )
)
group["n_obs_tot"] = group["n_obs_ztf"] + group["n_obs_lsst"]
group.sort_values("n_obs_tot", ascending=False)

,n_obs_lsst,n_days_lsst,n_obs_ztf,n_days_ztf,n_obs_tot
sso_name,,,,,
2000 YZ49,77.0,37.945895,203.0,1647.662789,280.0
2001 FA170,23.0,14.946738,221.0,2121.773993,244.0
2003 BU55,104.0,54.033942,128.0,2333.704005,232.0
2000 WX174,32.0,19.993534,187.0,2073.773692,219.0
2009 SV242,88.0,45.000101,100.0,1046.871539,188.0
2000 HN60,21.0,14.978665,162.0,1647.744803,183.0
2002 VF21,103.0,37.988796,75.0,2306.799005,178.0
2000 RA105,65.0,37.988796,109.0,1577.798623,174.0
2002 VK44,80.0,26.091979,91.0,2360.966204,171.0


[2009 SV242](https://lsst.fink-portal.org/K09SO2V) is a good candidate